In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import kagglehub
import tensorflow_text as text
import tf_keras as keras
import tensorflow_hub as hub

/Users/pernavjain/Documents/ML/CSTD/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.model_download("pernavjain/customer-support-dispatcher/keras/default")
print("Path to model files:", path)
model = keras.models.load_model(
    f"{path}/customer_support_ticket_dispatcher_v1.keras",
    custom_objects={"KerasLayer": hub.KerasLayer},
    compile=False
)

Path to model files: /Users/pernavjain/.cache/kagglehub/models/pernavjain/customer-support-dispatcher/keras/default/2


In [3]:
import tensorflow as tf
import numpy as np
import json

In [4]:
email = ["""Hi,

I’m not even sure which team this belongs to anymore. We renewed the subscription on Monday, but since yesterday half the users are being thrown out of the app after MFA, and the invoice also shows an extra charge I did not authorize. One of our admins said he got a login alert from a country we do not operate in, so I changed the password, but now the dashboard is also timing out on Chrome and Safari.

Please look into this urgently because we are blocked, and I need to know whether this is a billing mistake, a security issue, or some backend bug after the last update.

Thanks."""]

In [5]:
prediction = model.predict(email)
prediction

1/1 [==============================] - 0s 442ms/step


[array([[ 1.3224036 ,  0.37519398, -1.9110106 , -1.7666873 ,  2.544188  ]],
       dtype=float32),
 array([[0.8847246]], dtype=float32)]

In [6]:
prediction

[array([[ 1.3224036 ,  0.37519398, -1.9110106 , -1.7666873 ,  2.544188  ]],
       dtype=float32),
 array([[0.8847246]], dtype=float32)]

In [7]:
team_probs = tf.nn.softmax(prediction[0][0], axis=-1).numpy()
team_probs

array([0.20550655, 0.07969991, 0.00810162, 0.00935945, 0.6973325 ],
      dtype=float32)

In [8]:
predicted_class = np.argmax(team_probs)
predicted_class

np.int64(4)

In [9]:
urgency_score = prediction[1][0][0]
urgency_score

np.float32(0.8847246)

In [10]:
with open(f"{path}/id_to_team.json") as f:
    id_to_team = json.load(f)

In [11]:
predicted_team = id_to_team[f"{predicted_class}"]
confidence = team_probs[predicted_class]
print(predicted_team)
print(confidence)
print(urgency_score)

security
0.6973325
0.8847246
